In [1]:
!nvidia-smi

Wed May 13 13:04:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile main.cu

#include <iostream>
#include <cuda_runtime.h>

#define N 1000000
#define MATRIX_SIZE 512

using namespace std;

// Vector Addition Kernel
__global__ void vectorAdd(float* A, float* B, float* C, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if(i < n)
        C[i] = A[i] + B[i];
}

// Matrix Multiplication Kernel
__global__ void matrixMulKernel(float* A, float* B, float* C, int n)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < n && col < n)
    {
        float sum = 0;

        for(int i = 0; i < n; i++)
            sum += A[row * n + i] * B[i * n + col];

        C[row * n + col] = sum;
    }
}

int main()
{
    // VECTOR ADDITION
    float *A, *B, *C;
    float *d_A, *d_B, *d_C;

    size_t size = N * sizeof(float);

    A = new float[N];
    B = new float[N];
    C = new float[N];

    for(int i = 0; i < N; i++)
    {
        A[i] = i;
        B[i] = 2 * i;
    }

    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, size, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    cudaMemcpy(C, d_C, size, cudaMemcpyDeviceToHost);

    cout << "Vector Addition Output:" << endl;
    cout << "C[0] = " << C[0]
         << ", C[N-1] = " << C[N-1] << endl;

    // MATRIX MULTIPLICATION
    int matrixSize = MATRIX_SIZE * MATRIX_SIZE * sizeof(float);

    float *MA, *MB, *MC;
    float *d_MA, *d_MB, *d_MC;

    MA = new float[MATRIX_SIZE * MATRIX_SIZE];
    MB = new float[MATRIX_SIZE * MATRIX_SIZE];
    MC = new float[MATRIX_SIZE * MATRIX_SIZE];

    for(int i = 0; i < MATRIX_SIZE * MATRIX_SIZE; i++)
    {
        MA[i] = 1.0f;
        MB[i] = 2.0f;
    }

    cudaMalloc(&d_MA, matrixSize);
    cudaMalloc(&d_MB, matrixSize);
    cudaMalloc(&d_MC, matrixSize);

    cudaMemcpy(d_MA, MA, matrixSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_MB, MB, matrixSize, cudaMemcpyHostToDevice);

    dim3 threads(16,16);

    dim3 blocks((MATRIX_SIZE + 15)/16,
                (MATRIX_SIZE + 15)/16);

    matrixMulKernel<<<blocks, threads>>>(d_MA, d_MB,
                                         d_MC, MATRIX_SIZE);

    cudaMemcpy(MC, d_MC, matrixSize,
               cudaMemcpyDeviceToHost);

    cout << "\nMatrix Multiplication Output:" << endl;

    cout << "MC[0] = " << MC[0]
         << ", MC[last] = "
         << MC[MATRIX_SIZE*MATRIX_SIZE - 1]
         << endl;

    return 0;
}

Writing main.cu


In [3]:
!nvcc main.cu -o main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./main

Vector Addition Output:
C[0] = 0, C[N-1] = 3e+06

Matrix Multiplication Output:
MC[0] = 1024, MC[last] = 1024
